# Gibbs and Slice Sampling: Two Cures for Random-Walk MCMC

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/zhubarb/sesen_ai_ml_tutorials/blob/main/notebooks/bayesian/gibbs_slice_sampling.ipynb)

Companion notebook to the [blog post](https://sesen.ai/blog/gibbs-slice-sampling-cures-random-walk-mcmc). Three samplers in plain NumPy: random-walk Metropolis, Gibbs, and slice. Compared on a correlated bivariate Gaussian (Bishop §11.3 Fig 11.11), then slice on a 1D bimodal target.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
rng = np.random.default_rng(42)
plt.rcParams.update({'figure.figsize': (7, 4), 'axes.spines.top': False, 'axes.spines.right': False})


## 1. Gibbs sampling on a correlated bivariate Gaussian

Target: `N([0,0], Sigma)` with `Sigma = [[1, rho], [rho, 1]]`. The conditionals are themselves Gaussian:
`p(z1 | z2) = N(rho * z2, 1 - rho^2)` and symmetrically for `z2 | z1`. Each Gibbs step samples one coordinate from its conditional.


In [ ]:
def gibbs_bivariate_normal(n_cycles, rho, x0=None, rng=rng):
    sigma = np.sqrt(1 - rho ** 2)
    x = np.array([3.5, -3.5]) if x0 is None else np.asarray(x0, dtype=float).copy()
    trace = [x.copy()]
    for _ in range(n_cycles):
        x[0] = rho * x[1] + sigma * rng.standard_normal()
        trace.append(x.copy())
        x[1] = rho * x[0] + sigma * rng.standard_normal()
        trace.append(x.copy())
    return np.array(trace)

rho = 0.95
trace = gibbs_bivariate_normal(800, rho, x0=[3.5, -3.5])
print(f'samples: {len(trace)}, marginal mean = {trace[:, 0].mean():.3f} (target 0)')


In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
grid = np.linspace(-4, 4, 200)
X, Y = np.meshgrid(grid, grid)
cov = np.array([[1, rho], [rho, 1]])
cov_inv = np.linalg.inv(cov)
Z = np.exp(-0.5 * (X * cov_inv[0,0] * X + 2 * X * cov_inv[0,1] * Y + Y * cov_inv[1,1] * Y))
ax.contour(X, Y, Z, levels=6, alpha=0.6)
ax.plot(trace[:200, 0], trace[:200, 1], lw=0.6, alpha=0.7, color='C1')
ax.set_aspect('equal'); ax.set_xlim(-4, 4); ax.set_ylim(-4, 4)
ax.set_xlabel('z1'); ax.set_ylabel('z2'); ax.set_title(f'Gibbs trajectory, rho = {rho}')
plt.show()


## 2. Random-walk Metropolis on the same target

Step size matched to the smallest local standard deviation. Acceptance rate around 50% per usual MH heuristic.


In [ ]:
def metropolis_bivariate_normal(n_steps, rho, step_size, x0=None, rng=rng):
    cov = np.array([[1, rho], [rho, 1]])
    cov_inv = np.linalg.inv(cov)
    x = np.array([3.5, -3.5]) if x0 is None else np.asarray(x0, dtype=float).copy()
    log_p = lambda z: -0.5 * z @ cov_inv @ z
    trace = [x.copy()]
    log_p_curr = log_p(x)
    n_accept = 0
    for _ in range(n_steps):
        prop = x + step_size * rng.standard_normal(2)
        log_p_prop = log_p(prop)
        if np.log(rng.uniform()) < log_p_prop - log_p_curr:
            x = prop; log_p_curr = log_p_prop; n_accept += 1
        trace.append(x.copy())
    return np.array(trace), n_accept / n_steps

mh_trace, accept = metropolis_bivariate_normal(1600, rho, step_size=2.0 * np.sqrt(1 - rho))
print(f'MH acceptance: {accept:.1%}')


## 3. Mixing-time scaling: Gibbs ESS vs correlation rho

Bishop's `(L/l)^2 ~ 1/(1-rho)` scaling: the autocorrelation of `z1` over Gibbs cycles decays slower as rho approaches 1.


In [ ]:
def autocorr(x, max_lag):
    x = x - x.mean(); var = np.var(x)
    if var == 0: return np.zeros(max_lag + 1)
    out = np.empty(max_lag + 1); n = len(x)
    for lag in range(max_lag + 1):
        out[lag] = np.dot(x[:n - lag], x[lag:]) / ((n - lag) * var)
    return out

fig, ax = plt.subplots()
for rho in [0.0, 0.5, 0.9, 0.99]:
    trace = gibbs_bivariate_normal(5000, rho)
    cycle = trace[::2]
    ac = autocorr(cycle[:, 0], 60)
    ax.plot(ac, lw=2, label=f'rho = {rho}')
ax.axhline(0, color='gray', lw=0.5)
ax.set_xlabel('lag (cycles)'); ax.set_ylabel('autocorrelation of z1')
ax.set_title('Gibbs autocorrelation grows with target correlation')
ax.legend(); plt.show()


## 4. Slice sampling on a bimodal 1D target

Algorithm: at each step, draw `u ~ Uniform(0, p_unnorm(z))`, then sample `z` uniformly from the slice `{z : p_unnorm(z) > u}` via stepping-out and shrinking.


In [ ]:
def bimodal_pdf_unnorm(z):
    return 0.5 * np.exp(-0.5 * (z - (-2))**2 / 0.7**2) + 0.5 * np.exp(-0.5 * (z - 2)**2 / 0.7**2)

def slice_sampler_1d(p_unnorm, x0, n_steps, w=2.0, rng=rng):
    x = float(x0)
    trace = [x]
    for _ in range(n_steps):
        u = rng.uniform(0, p_unnorm(x))
        L = x - w * rng.uniform(); R = L + w
        while p_unnorm(L) > u: L -= w
        while p_unnorm(R) > u: R += w
        while True:
            z_new = rng.uniform(L, R)
            if p_unnorm(z_new) > u:
                x = z_new; break
            (L, R) = (z_new, R) if z_new < x else (L, z_new)
        trace.append(x)
    return np.array(trace)

samples = slice_sampler_1d(bimodal_pdf_unnorm, x0=-3.0, n_steps=4000, w=1.5)
fig, ax = plt.subplots()
z_grid = np.linspace(-5, 5, 400)
p_curve = bimodal_pdf_unnorm(z_grid)
ax.plot(z_grid, p_curve / np.trapezoid(p_curve, z_grid), lw=2, label='target (normalised)')
ax.hist(samples[200:], bins=60, density=True, alpha=0.6, label='slice samples')
ax.set_xlabel('z'); ax.set_ylabel('density')
ax.set_title('Slice sampler on a bimodal target'); ax.legend(); plt.show()


## Exercises

1. **Block Gibbs.** Generalise `gibbs_bivariate_normal` to a 4D Gaussian with two correlated pairs. Implement two strategies: per-coordinate Gibbs and block Gibbs (sample each pair jointly). Compare ESS at rho = 0.95 within blocks.

2. **Componentwise slice on bivariate Gaussian.** Implement a 2D slice sampler that updates one coordinate at a time using the 1D slice sampler. Run it on the rho = 0.95 bivariate Gaussian and compare ESS to Gibbs.

3. **Effect of slice w.** Run the bimodal slice sampler with `w in [0.1, 1.0, 5.0, 50.0]`. Time each run and count step-out plus shrink iterations per sample. What's the optimal w in terms of total iterations?

4. **Over-relaxation.** Implement Adler's over-relaxation update (Bishop eq. 11.50) for the bivariate Gaussian Gibbs sampler. Sweep alpha in [-0.99, -0.5, 0, 0.5, 0.99] and plot the autocorrelation curves.

5. **Multimodal Gibbs failure.** Construct a 2D target that is two well-separated Gaussian clusters connected only by a thin ridge of low (but positive) density. Run a Gibbs sampler. How many cycles before the chain crosses between modes?


## References

- Bishop, C. M. (2006). *Pattern Recognition and Machine Learning*, §11.3 and §11.4.
- Geman, S., & Geman, D. (1984). Stochastic relaxation, Gibbs distributions, and the Bayesian restoration of images. *IEEE TPAMI*, 6(6), 721-741.
- Neal, R. M. (2003). Slice sampling. *Annals of Statistics*, 31(3), 705-767.
- Adler, S. L. (1981). Over-relaxation method for the Monte Carlo evaluation of the partition function for multiquadratic actions. *Phys. Rev. D*, 23(12), 2901-2904.
- Jensen, C. S., Kong, A., & Kjærulff, U. (1995). Blocking Gibbs sampling in very large probabilistic expert systems. *International Journal of Human-Computer Studies*, 42(6), 647-666.
